In [2]:
"""
✔ Detectar automaticamente handle na URL
✔ Criar logs de erros
✔ Baixar apenas PDFs que ainda não existem
✔ Baixar usando retry automático
✔ Extração automática do link PDF de páginas HTML (DSpace / OJS / Dataverse)
✔ NÃO baixar novamente se o PDF já existir
✔ Nomear cada PDF com uma hash de 16 dígitos baseada na URL
✔ Ler as URLs do arquivo https://jupyter.brapci.inf.br/hub/user-redirect/lab/tree/EBBC/teste_juliana/ppgsCSA_ano_PPG_link.csv
✔ Salvar em data/pdf/
 """

'\n✔ Detectar automaticamente handle na URL\n✔ Criar logs de erros\n✔ Baixar apenas PDFs que ainda não existem\n✔ Baixar usando retry automático\n✔ Extração automática do link PDF de páginas HTML (DSpace / OJS / Dataverse)\n✔ NÃO baixar novamente se o PDF já existir\n✔ Nomear cada PDF com uma hash de 16 dígitos baseada na URL\n✔ Ler as URLs do arquivo https://jupyter.brapci.inf.br/hub/user-redirect/lab/tree/EBBC/teste_juliana/ppgsCSA_ano_PPG_link.csv\n✔ Salvar em data/pdf/\n '

In [3]:
import os
import requests
import hashlib

In [4]:
def hash16(texto: str) -> str:
    """
    Gera uma hash MD5 de 16 dígitos baseada em um texto (URL).
    """
    return hashlib.md5(texto.encode("utf-8")).hexdigest()[:16]

In [6]:
import csv
from collections import defaultdict

arquivo_entrada = "ppgsCSA_ano_PPG_link.csv"

# dicionário: ano -> lista de links
links_por_ano = defaultdict(list)

with open(arquivo_entrada, newline='', encoding='utf-8') as f:
    leitor = csv.DictReader(f)
    for linha in leitor:
        ano = linha['ano']
        link = linha['link']
        links_por_ano[ano].append(link)

# salvar um CSV por ano
for ano, links in links_por_ano.items():
    nome_saida = f"ppgsCSA_{ano}_link.csv"
    with open(nome_saida, 'w', newline='', encoding='utf-8') as f:
        escritor = csv.writer(f)
        escritor.writerow(['link'])
        for link in links:
            escritor.writerow([link])

    print(f"Salvo: {nome_saida}")



Salvo: ppgsCSA_2025_link.csv
Salvo: ppgsCSA_2024_link.csv
Salvo: ppgsCSA_2022_link.csv
Salvo: ppgsCSA_2023_link.csv
Salvo: ppgsCSA_2021_link.csv
Salvo: ppgsCSA_2020_link.csv


In [7]:
import csv
from collections import defaultdict

arquivo_entrada = "ppgsCSA_ano_PPG_link.csv"

# dicionário: ano -> lista de links
links_por_ano = defaultdict(list)

with open(arquivo_entrada, newline='', encoding='utf-8') as f:
    leitor = csv.DictReader(f)
    for linha in leitor:
        ano = linha['ano']
        link = linha['link']
        if link:  # evita linhas vazias
            links_por_ano[ano].append(link)

# salvar um TXT por ano
for ano, links in links_por_ano.items():
    nome_saida = f"ppgsCSA_{ano}_link.txt"
    with open(nome_saida, 'w', encoding='utf-8') as f:
        for link in links:
            f.write(link + "\n")

    print(f"Salvo: {nome_saida}")


Salvo: ppgsCSA_2025_link.txt
Salvo: ppgsCSA_2024_link.txt
Salvo: ppgsCSA_2022_link.txt
Salvo: ppgsCSA_2023_link.txt
Salvo: ppgsCSA_2021_link.txt
Salvo: ppgsCSA_2020_link.txt


In [11]:
import os
import requests
import hashlib

def hash16(texto):
    return hashlib.md5(texto.encode("utf-8")).hexdigest()[:16]

def baixar_pdfs_do_corpus(
    arquivo_corpus="ppgsCSA_2020_link.txt",
    pasta_pdf="data2020/pdf"
):
    """
    Lê o arquivo TXT com links, baixa os PDFs e salva na pasta indicada,
    usando hash de 16 dígitos como nome do arquivo.
    """

    # Criar pasta de destino
    os.makedirs(pasta_pdf, exist_ok=True)

    # Ler lista de URLs
    with open(arquivo_corpus, "r", encoding="utf-8") as f:
        urls = [linha.strip() for linha in f if linha.strip()]

    # Processar linha por linha
    for url in urls:
        print(f"\n🔎 URL: {url}")

        nome_pdf = hash16(url) + ".pdf"
        caminho_pdf = os.path.join(pasta_pdf, nome_pdf)

        if os.path.exists(caminho_pdf):
            print(f"⏩ Arquivo já existe, pulando")
            continue

        try:
            print("⬇️ Baixando...")
            resposta = requests.get(url, timeout=30)

            if resposta.status_code == 200 and resposta.content[:4] == b"%PDF":
                with open(caminho_pdf, "wb") as f:
                    f.write(resposta.content)
                print(f"✅ PDF salvo")
            else:
                print(f"⚠️ Não é PDF válido (status {resposta.status_code})")

        except Exception as e:
            print(f"❌ Erro ao baixar: {e}")

    print("\n🎉 Processo concluído!")


In [12]:
baixar_pdfs_do_corpus()


🔎 URL: http://www.bibliotecadigital.ufrgs.br/da.php?nrb=001117299&loc=2020&l=eec58ceb68418349
⬇️ Baixando...
✅ PDF salvo

🔎 URL: http://www.bibliotecadigital.ufrgs.br/da.php?nrb=001124769&loc=2021&l=4102b4a3df7d6878
⬇️ Baixando...
✅ PDF salvo

🔎 URL: http://www.bibliotecadigital.ufrgs.br/da.php?nrb=001123248&loc=2021&l=58781074161160d4
⬇️ Baixando...
✅ PDF salvo

🔎 URL: http://www.bibliotecadigital.ufrgs.br/da.php?nrb=001118881&loc=2020&l=a527267146e1e9ed
⬇️ Baixando...
✅ PDF salvo

🔎 URL: http://www.bibliotecadigital.ufrgs.br/da.php?nrb=001125138&loc=2021&l=fa3dca8800d5e92c
⬇️ Baixando...
✅ PDF salvo

🔎 URL: http://www.bibliotecadigital.ufrgs.br/da.php?nrb=001116289&loc=2020&l=bd443324b9e029b1
⬇️ Baixando...
✅ PDF salvo

🔎 URL: http://www.bibliotecadigital.ufrgs.br/da.php?nrb=001127956&loc=2025&l=657f1e51f465bc25
⬇️ Baixando...
✅ PDF salvo

🔎 URL: http://www.bibliotecadigital.ufrgs.br/da.php?nrb=001127440&loc=2021&l=28d0fd33824255f1
⬇️ Baixando...
✅ PDF salvo

🔎 URL: http://www.bibli

In [13]:
import os
import requests
import hashlib

def hash16(texto):
    return hashlib.md5(texto.encode("utf-8")).hexdigest()[:16]


In [14]:
def baixar_pdfs_do_corpus(arquivo_corpus, pasta_pdf):
    """
    Lê um arquivo TXT com links, baixa os PDFs e salva na pasta indicada,
    usando hash de 16 dígitos como nome do arquivo.
    """

    os.makedirs(pasta_pdf, exist_ok=True)

    with open(arquivo_corpus, "r", encoding="utf-8") as f:
        urls = [linha.strip() for linha in f if linha.strip()]

    for url in urls:
        print(f"\n🔎 {url}")

        nome_pdf = hash16(url) + ".pdf"
        caminho_pdf = os.path.join(pasta_pdf, nome_pdf)

        if os.path.exists(caminho_pdf):
            print("⏩ Já existe, pulando")
            continue

        try:
            resposta = requests.get(url, timeout=30)

            if resposta.status_code == 200 and resposta.content[:4] == b"%PDF":
                with open(caminho_pdf, "wb") as f:
                    f.write(resposta.content)
                print("✅ PDF salvo")
            else:
                print(f"⚠️ Não retornou PDF válido (status {resposta.status_code})")

        except Exception as e:
            print(f"❌ Erro: {e}")

    print(f"\n🎉 Concluído: {pasta_pdf}")


In [15]:
anos = [2021, 2022, 2023, 2024, 2025]

for ano in anos:
    arquivo_txt = f"ppgsCSA_{ano}_link.txt"
    pasta_destino = f"data{ano}/pdf"

    print(f"\n==============================")
    print(f"📅 Processando ano {ano}")
    print(f"==============================")

    baixar_pdfs_do_corpus(
        arquivo_corpus=arquivo_txt,
        pasta_pdf=pasta_destino
    )



📅 Processando ano 2021

🔎 http://www.bibliotecadigital.ufrgs.br/da.php?nrb=001137425&loc=2022&l=616475adc374cdaa
✅ PDF salvo

🔎 http://www.bibliotecadigital.ufrgs.br/da.php?nrb=001136112&loc=2022&l=ff502bb297eb8a9d
✅ PDF salvo

🔎 http://www.bibliotecadigital.ufrgs.br/da.php?nrb=001126745&loc=2021&l=f61be9c3113362bd
✅ PDF salvo

🔎 http://www.bibliotecadigital.ufrgs.br/da.php?nrb=001126379&loc=2021&l=2a7266925b59948c
✅ PDF salvo

🔎 http://www.bibliotecadigital.ufrgs.br/da.php?nrb=001135171&loc=2021&l=5ca933ec88476fdc
✅ PDF salvo

🔎 http://www.bibliotecadigital.ufrgs.br/da.php?nrb=001138604&loc=2022&l=4be244fea2dc27a1
✅ PDF salvo

🔎 http://www.bibliotecadigital.ufrgs.br/da.php?nrb=001133292&loc=2021&l=e69d4f6aeddf1d07
✅ PDF salvo

🔎 http://www.bibliotecadigital.ufrgs.br/da.php?nrb=001128507&loc=2021&l=5a36df63d8e08351
✅ PDF salvo

🔎 http://www.bibliotecadigital.ufrgs.br/da.php?nrb=001134870&loc=2021&l=f27ef66409b73f3b
✅ PDF salvo

🔎 http://www.bibliotecadigital.ufrgs.br/da.php?nrb=001135

In [1]:
import os
import requests
import hashlib

def hash16(texto):
    return hashlib.md5(texto.encode("utf-8")).hexdigest()[:16]


def baixar_pdfs_do_corpus(arquivo_corpus, pasta_pdf, arquivo_erros):
    """
    Baixa PDFs a partir de um TXT com links.
    Salva PDFs por hash e registra erros em arquivo TXT.
    """

    os.makedirs(pasta_pdf, exist_ok=True)

    with open(arquivo_corpus, "r", encoding="utf-8") as f:
        urls = [linha.strip() for linha in f if linha.strip()]

    for url in urls:
        nome_pdf = hash16(url) + ".pdf"
        caminho_pdf = os.path.join(pasta_pdf, nome_pdf)

        if os.path.exists(caminho_pdf):
            continue

        try:
            resposta = requests.get(url, timeout=30)

            if resposta.status_code == 200 and resposta.content[:4] == b"%PDF":
                with open(caminho_pdf, "wb") as f:
                    f.write(resposta.content)
            else:
                with open(arquivo_erros, "a", encoding="utf-8") as log:
                    log.write(
                        f"{url}\tSTATUS:{resposta.status_code}\tNAO_PDF\n"
                    )

        except Exception as e:
            with open(arquivo_erros, "a", encoding="utf-8") as log:
                log.write(
                    f"{url}\tERRO:{str(e)}\n"
                )


In [2]:
anos = [2021, 2022, 2023, 2024, 2025]

for ano in anos:
    arquivo_txt = f"ppgsCSA_{ano}_link.txt"
    pasta_destino = f"data{ano}/pdf"
    arquivo_erros = f"data{ano}/erros_{ano}.txt"

    print(f"📅 Processando {ano}")

    baixar_pdfs_do_corpus(
        arquivo_corpus=arquivo_txt,
        pasta_pdf=pasta_destino,
        arquivo_erros=arquivo_erros
    )


📅 Processando 2021
📅 Processando 2022
📅 Processando 2023
📅 Processando 2024
📅 Processando 2025
